# DPL001-05 集保餘額查詢


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import duckdb

In [ ]:
from finlab import data

In [ ]:
# 引用自建公用模組
sys.path.insert(0, str(Path.cwd().parent))
from proj_util_pkg.settings import ProjEnvSettings

from proj_util_pkg.finlab_api import finlab_manager as flm
from proj_util_pkg.common.duckdb_tool import insert_dataframe_to_duckdb

## 公用參數設定

In [ ]:
# 欄數全展開
pd.set_option("display.max_columns", None)

In [ ]:
# finlab api 服務初始化
finlab = flm.FinlabManager()
data.force_cloud_download = False

## Function Block  


## 外部資料讀取  

In [ ]:
# 讀取台股集保餘額資訊
tdcc_balance = data.get("inventory", save_to_storage=True)

In [ ]:
tdcc_balance.tail(10)

## 資料留存ＤＢ

In [ ]:
# 設定資料庫路徑
TWSTOCK_DATA_ROOT = os.environ.get("hist_data_path")
twstock_db_path = f"{TWSTOCK_DATA_ROOT}/twstock.duckdb"

In [ ]:
# 連線資料庫
conn_duckdb = duckdb.connect(twstock_db_path)

In [ ]:
table_name = "tdcc_balance"

In [ ]:
insert_row_count = insert_dataframe_to_duckdb(
    conn_duckdb, 
    tdcc_balance, 
    table_name, 
    date_column='date'
)

f"成功插入 {insert_row_count} 筆資料到 {table_name}"

In [ ]:
# 查詢表中所有資料
conn_duckdb.execute(f"SELECT * FROM {table_name} order by date desc LIMIT 10").fetch_df()

In [ ]:
# 關閉資料庫連線
conn_duckdb.close()
print("資料庫連線已關閉")